### Our defined Specialty Buckets

In [0]:
-- All the remaining specialties not present here but present in data are clubbed into Others

CREATE OR REPLACE TEMPORARY VIEW specialty_groupings AS
SELECT * FROM (
    VALUES
        ('Psychiatry and Neurology', 'Psychiatry & Neurology'), 
        ('Psychiatry and Neurology', 'Neurological Surgery'), 
        ('Others', 'Surgery'), 
        ('Geneticist', 'Medical Genetics'), 
        ('Pediatrician', 'Pediatrics'), 
        ('PCP', 'Family Medicine'), 
        ('PCP', 'Internal Medicine'), 
        ('NP/PA', 'Physician Assistant'), 
        ('NP/PA', 'Nurse Practitioner'), 
        ('Others', 'Orthopaedic Surgery'), 
        ('Others', 'Otolaryngology'), 
        ('Others', 'Ophthalmology'), 
        ('NP/PA', 'Nurse Anesthetist, Certified Registered'), 
        ('Others', 'Anesthesiology'), 
        ('Others', 'Radiology'), 
        ('Others', 'Pathology'), 
        ('Others', 'Emergency Medicine'), 
        ('Others', 'Hospitalist'), 
        ('Others', 'Physical Medicine & Rehabilitation')
) AS t(mapped_bucket, specialty);

In [0]:
-- -- Using all codes (2 years)
-- create or replace temporary view mpsii_treatment_table as
-- SELECT *
--     FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
--                  COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
--                  NDC11 AS CODE,
--                  MEDICAL_EVENT_ID AS EVENT_ID,
--                  SERVICE_DATE AS FILL_DATE,
--                  PLACE_OF_SERVICE,
--                  KH_PLAN_ID AS KH_PLAN,
--                 'MEDICAL_EVENTS' AS TABLE_NAME

--          FROM com_edp_prd.com_raw.kom_medical_events
--          WHERE NDC11 IN ('54092070001','540920700')
--     UNION
--          SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
--                  PRESCRIBER_NPI AS NPI,
--                  NDC11 AS CODE,
--                  PHARMACY_EVENT_ID as EVENT_ID,
--                  FILL_DATE,
--                  NULL AS PLACE_OF_SERVICE,
--                  COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
--                  'PHARMACY_EVENTS' AS TABLE_NAME
--            FROM com_edp_prd.com_raw.kom_pharmacy_events
--            WHERE NDC11 IN ('54092070001','540920700')
--            AND TRANSACTION_RESULT = 'PAID'
--     UNION

--          SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
--                 RENDERING_NPI AS NPI,
--                 PROCEDURE_CODE AS CODE,   
--                 MEDICAL_EVENT_ID AS EVENT_ID,
--                 SERVICE_DATE AS FILL_DATE, 
--                 PLACE_OF_SERVICE,
--                 KH_PLAN_ID AS KH_PLAN,
--                 'MEDICAL_EVENTS' AS TABLE_NAME


--            FROM com_edp_prd.com_raw.kom_medical_events 
--            WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
--           --  WHERE PROCEDURE_CODE IN ('J1743')
-- )
-- WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31';

In [0]:
-- Using all codes (2 years)
create or replace temporary view mpsii_treatment_table as
SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS ndc,
                 procedure_code,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS ndc,
                 null as procedure_code,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                NDC11 AS ndc,
                procedure_code, 
                PROCEDURE_CODE AS CODE,  
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
          --  WHERE PROCEDURE_CODE IN ('J1743')
)
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31';

In [0]:
select * from mpsii_treatment_table limit 3

In [0]:
create or replace temporary view mpsii_diagnosis_table as 
with mpsii_1dx_specified as (
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      NDC11 AS ndc,
      procedure_code,  
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      NDC11 AS ndc,
      null as procedure_code,  
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
mpsii_2dx_specified as (
  select *
  from mpsii_1dx_specified
  where patient_id in (select a.patient_id from mpsii_1dx_specified as a group by a.patient_id having count(distinct a.fill_date) >= 2)
),
mpsii_1dx_unspecified as (
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      NDC11 AS ndc,
      procedure_code,  
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      NDC11 AS ndc,
      null as procedure_code,  
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-08-01' AND '2025-07-31'
),
mpsii_2dx_unspecified as (
  select *
  from mpsii_1dx_unspecified
  where patient_id in (select distinct a.patient_id from mpsii_1dx_unspecified as a group by a.patient_id having count(distinct a.fill_date) >= 2) 
),
mpsii_2dx_specified_tx as (
  select *
  from mpsii_treatment_table where patient_id in (select distinct a.patient_id from mpsii_2dx_specified as a)
),
incremental_patient as (
  select distinct patient_id
  from mpsii_2dx_unspecified
  where patient_id in (select distinct a.patient_id from mpsii_treatment_table as a where a.code in ('54092070001','540920700','J1743'))
  and patient_id not in (select distinct b.patient_id from mpsii_2dx_specified_tx as b)
),
all_dx_patients_claims as (
  select * from mpsii_2dx_specified
  union
  select * from mpsii_1dx_unspecified where patient_id in (select distinct a.patient_id from incremental_patient as a) 
)
select *
from all_dx_patients_claims

In [0]:
select * from mpsii_diagnosis_table limit 3

In [0]:
select count(distinct patient_id), count(distinct MEDICAL_EVENT_ID) 
-- from mpsii_treatment_table
FROM com_edp_prd.com_raw.kom_medical_events
where PROCEDURE_CODE in ('J3490', 'J3590') 
-- and patient_id in (select distinct patient_id from mpsii_diagnosis_table) 
and SERVICE_DATE between '2023-08-01' and '2025-07-31'

In [0]:
with icd_descriptions AS (
    SELECT claim_code, description
    FROM (
        SELECT claim_code, 
               description,
               ROW_NUMBER() OVER (PARTITION BY claim_code ORDER BY source_priority) AS rn
        FROM (
            SELECT claim_code, description, 1 AS source_priority
            FROM com_edp_prd.cmpa_insights_internal_schema.icd_lookup_update
            UNION ALL
            SELECT claim_code, description, 2 AS source_priority
            FROM com_edp_prd.cmpa_insights_internal_schema.code_reference
            UNION ALL
            SELECT claim_code, description, 3 AS source_priority
            FROM com_edp_prd.cmpa_insights_internal_schema.code_reference_2
        )
    )
    WHERE rn = 1
)
select distinct  a.procedure_code, b.description, count(distinct a.patient_id) as patient_volume
from mpsii_treatment_table as a
left join icd_descriptions as b on a.procedure_code = b.claim_code
where a.procedure_code is not null and a.procedure_code ilike '%J%' and a.patient_id in (select distinct patient_id from mpsii_diagnosis_table)
group by 1,2 order by 3 desc

In [0]:
with treatment_table as (
  SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS ndc,
                 procedure_code,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events
        --  WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS ndc,
                 null as procedure_code,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
          --  WHERE NDC11 IN ('54092070001','540920700')
           where TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                NDC11 AS ndc,
                procedure_code, 
                PROCEDURE_CODE AS CODE,  
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
          --  WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
          --  WHERE PROCEDURE_CODE IN ('J1743')
)
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
),
mpsii_diagnosed_receiving_treatment_not_elaprase_v1 as (
  select * from treatment_table
  where patient_id in (select distinct patient_id from mpsii_diagnosis_table) and patient_id not in (select distinct patient_id from mpsii_treatment_table)
),
icd_descriptions AS (
    SELECT claim_code, description
    FROM (
        SELECT claim_code, 
               description,
               ROW_NUMBER() OVER (PARTITION BY claim_code ORDER BY source_priority) AS rn
        FROM (
            SELECT claim_code, description, 1 AS source_priority
            FROM com_edp_prd.cmpa_insights_internal_schema.icd_lookup_update
            UNION ALL
            SELECT claim_code, description, 2 AS source_priority
            FROM com_edp_prd.cmpa_insights_internal_schema.code_reference
            UNION ALL
            SELECT claim_code, description, 3 AS source_priority
            FROM com_edp_prd.cmpa_insights_internal_schema.code_reference_2
        )
    )
    WHERE rn = 1
)
select distinct  a.procedure_code, b.description, count(distinct a.patient_id) as patient_volume
from mpsii_diagnosed_receiving_treatment_not_elaprase_v1 as a
left join icd_descriptions as b on a.procedure_code = b.claim_code
where a.procedure_code is not null and a.procedure_code ilike '%J%'
group by 1,2 order by 3 desc

In [0]:
select distinct * from  mpsii_treatment_table

In [0]:
select distinct * from  mpsii_treatment_table where patient_id in (select distinct patient_id from mpsii_diagnosis_table)

In [0]:
select * from mpsii_treatment_table_new where patient_id in (select distinct patient_id from mpsii_diagnosis_table)

### After doing Primary NPI Tagging

In [0]:
create or replace temporary view elaprase_treated as 
with elaprase_treated_v1 as (
  select *
  from mpsii_treatment_table
),
pulling_specialities as (
  select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (select * from elaprase_treated_v1) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi
),
elaprase_treated_v2 as (
  select distinct n_pats as patient_id, npi 
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM pulling_specialities
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
)
select * from elaprase_treated_v2
where npi is not null;

In [0]:
create or replace temporary view mpsii_diagnosed as 
with mpsii_diagnosis_v1 as (
  select *
from mpsii_diagnosis_table
),
pulling_specialities as (
  select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (select * from mpsii_diagnosis_v1) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi
),
mpsii_diagnosis_v2 as (
  select distinct n_pats as patient_id, npi 
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM pulling_specialities
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
)
select * from mpsii_diagnosis_v2
where npi is not null;

In [0]:
select count(distinct npi) as elaprase_treater_hcp from mpsii_treatment_table

In [0]:
with mpsii_diagnosed_elaprase_treated_v1 as (
  select *
from mpsii_treatment_table
where patient_id in (select distinct patient_id from mpsii_diagnosis_table))
select npi, count(distinct patient_id)
from mpsii_diagnosed_elaprase_treated_v1
group by 1 order by 2 desc

In [0]:
with mpsii_diagnosed_elaprase_treated_v1 as (
  select *
from mpsii_treatment_table
where patient_id in (select distinct patient_id from mpsii_diagnosis_table))
select distinct npi
from mpsii_diagnosed_elaprase_treated_v1
where npi is not null

In [0]:
create or replace temporary view mpsii_diagnosed_elaprase_treated as 
with mpsii_diagnosed_elaprase_treated_v1 as (
  select *
from mpsii_treatment_table
where patient_id in (select distinct patient_id from mpsii_diagnosis_table)
),
pulling_specialities as (
  select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (select * from mpsii_diagnosed_elaprase_treated_v1) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi
),
mpsii_diagnosed_elaprase_treated_v2 as (
  select distinct n_pats as patient_id, npi 
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM pulling_specialities
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
)
select * from mpsii_diagnosed_elaprase_treated_v2
where npi is not null;

In [0]:
create or replace temporary view mpsii_diagnosed_receiving_treatment_not_elaprase as 
with treatment_table as (
  SELECT *
    FROM (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
        --  WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
          --  WHERE NDC11 IN ('54092070001','540920700')
           where TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
          --  WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
)
WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
),
mpsii_diagnosed_receiving_treatment_not_elaprase_v1 as (
  select * from treatment_table
  where patient_id in (select distinct patient_id from mpsii_diagnosis_table) and patient_id not in (select distinct patient_id from mpsii_treatment_table)
),
pulling_specialities as (
  select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (select * from mpsii_diagnosed_receiving_treatment_not_elaprase_v1) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi
),
mpsii_diagnosed_receiving_treatment_not_elaprase_v2 as (
  select distinct n_pats as patient_id, npi 
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM pulling_specialities
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
)
select * from mpsii_diagnosed_receiving_treatment_not_elaprase_v2
where npi is not null;

In [0]:
create or replace temporary view elaprase_treated_procdna_pov as 
with elaprase_treated_procdna_pov_v1 as (
  select *
from mpsii_treatment_table
where patient_id in (select distinct patient_id
from (select distinct patient_id
from mpsii_treatment_table 
where patient_id in (select distinct patient_id from mpsii_diagnosis_table)
union
select distinct patient_id
from mpsii_treatment_table
where code in ('54092070001','540920700','J1743')))
),
pulling_specialities as (
  select a.*,
        b.HCO_PRIMARY_NPI,
        b.PRIMARY_SPECIALTY,
        b.SECONDARY_SPECIALTY,
        CASE 
        WHEN primary_specialty like '%Genetic%' or secondary_specialty like '%Genetic%' THEN 'Geneticist'
        WHEN primary_specialty like '%Pediatrics%' THEN 'Pediatrician'
        WHEN primary_specialty like '%Psychiatry & Neurology%' or secondary_specialty like '%Neurodevelopmental Disabilities%' or 
        primary_specialty like '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
        WHEN primary_specialty like '%Nurse Practitioner%' or primary_specialty like '%Physician Assistant%' THEN 'NPPA'
        WHEN primary_specialty like '%Internal Medicine%' or secondary_specialty like '%Internal Medicine%' THEN 'PCP'
        WHEN primary_specialty like '%Family Medicine%' or secondary_specialty like '%Family Medicine%' THEN 'PCP'
        When a.npi is null then 'NA'
        ELSE 'Others'
    END AS SPECIALTY from (select * from elaprase_treated_procdna_pov_v1) a left join com_edp_prd.com_raw.kom_providers b
on a.npi= b.npi
),
elaprase_treated_procdna_pov_v2 as (
  select distinct n_pats as patient_id, npi 
from (SELECT DISTINCT 
    NPI, 
    SPECIALTY,
    PATIENT_ID AS N_PATS, 
    Fill_date,
    KH_PLAN,
    HCO_PRIMARY_NPI,
    PLACE_OF_SERVICE,
    'TX' as PATIENT_TYPE
FROM (
    SELECT DISTINCT 
        PATIENT_ID,
        Fill_Date,
        KH_PLAN,
        HCO_PRIMARY_NPI,
        PLACE_OF_SERVICE,
        NPI, 
        SPECIALTY, 
        FINAL_HCP_RANK
    FROM (
        SELECT *, 
               DENSE_RANK() OVER (PARTITION BY PATIENT_ID ORDER BY HCP_RANK_1) AS FINAL_HCP_RANK
        FROM (
            SELECT *, 
                   MIN(HCP_RANK) OVER (PARTITION BY PATIENT_ID, NPI) AS HCP_RANK_1
            FROM (
                SELECT *, 
                       RANK() OVER (PARTITION BY PATIENT_ID ORDER BY PRIORITY, NO_OF_VISITS DESC, DATE(FILL_DATE) DESC,NPI) AS HCP_RANK
                FROM (
                    -- Precompute NO_OF_VISITS using a subquery
                    SELECT 
                        PATIENT_ID,
                        NPI,
                        SPECIALTY,
                        FILL_DATE,
                        KH_PLAN,
                        HCO_PRIMARY_NPI,
                        PLACE_OF_SERVICE,
                        PRIORITY,
                        NO_OF_VISITS
                    FROM (
                        SELECT 
                            PATIENT_ID,
                            NPI,
                            SPECIALTY,
                            FILL_DATE,
                            KH_PLAN,
                            HCO_PRIMARY_NPI,
                            PLACE_OF_SERVICE,
                            CASE 
                                WHEN SPECIALTY = 'Geneticist' THEN 1 
                                WHEN SPECIALTY = 'Psychiatry & Neurology' THEN 2 
                                WHEN SPECIALTY = 'Pediatrician' THEN 3 
                                WHEN SPECIALTY = 'PCP' THEN 4 
                                WHEN SPECIALTY = 'NPPA' THEN 5
                                WHEN SPECIALTY = 'Others' THEN 6 
                                ELSE 7
                            END AS PRIORITY,
                            COUNT(DISTINCT FILL_DATE) AS NO_OF_VISITS
                        FROM pulling_specialities
                        GROUP BY PATIENT_ID, NPI, SPECIALTY, FILL_DATE,KH_PLAN,HCO_PRIMARY_NPI,PLACE_OF_SERVICE
                    )
                )
            )
        ) 
    ) 
    WHERE HCP_RANK = 1
))
)
select * from elaprase_treated_procdna_pov_v2
where npi is not null;

In [0]:
select npi, count(distinct patient_id) from elaprase_treated_procdna_pov
group by 1 order by 2 desc

In [0]:
create or replace temporary view hcp_universe as
select distinct npi from (
  select distinct npi 
from elaprase_treated
union
select distinct npi 
from mpsii_diagnosed
union
select distinct npi 
from mpsii_diagnosed_elaprase_treated
union
select distinct npi
from mpsii_diagnosed_receiving_treatment_not_elaprase
union
select distinct npi
from (with mpsii_diagnosed_elaprase_treated_v1 as (
  select *
from mpsii_treatment_table
where patient_id in (select distinct patient_id from mpsii_diagnosis_table))
select distinct npi
from mpsii_diagnosed_elaprase_treated_v1
where npi is not null)
)

### Pulling Numbers

In [0]:
with elaprase_treatment as (
  select npi, count(distinct patient_id) as elaprase_treated_patients
  from elaprase_treated group by 1 order by 2 desc
),
mpsii_diagnosed as (
  select npi, count(distinct patient_id) as mpsii_diagnosed_patients
  from mpsii_diagnosed group by 1 order by 2 desc
),
mpsii_treated_not_elaprase as (
  select npi, count(distinct patient_id) as mpsii_treated_patients_not_elaprase
  from mpsii_diagnosed_receiving_treatment_not_elaprase group by 1 order by 2 desc  
),
mpsii_elaprase_treated as (
  select npi, count(distinct patient_id) as mpsii_elaprase_treated_patients
  from mpsii_diagnosed_elaprase_treated group by 1 order by 2 desc
),
joining_patient_numbers as (
  select a.*, b.elaprase_treated_patients, c.mpsii_diagnosed_patients, d.mpsii_treated_patients_not_elaprase, e.mpsii_elaprase_treated_patients
  from hcp_universe as a
  left join elaprase_treatment as b on a.npi = b.npi
  left join mpsii_diagnosed as c on a.npi = c.npi
  left join mpsii_treated_not_elaprase as d on a.npi = d.npi
  left join mpsii_elaprase_treated as e on a.npi = e.npi
),
kol_list AS (
  SELECT * FROM VALUES
    ('1699743088'),('1154431567'),('1467848366'),('1528585833'),('1114949617'),('1861866717'),('1942545314'),('1255435301'),('1104395656'),('1770949901'),('1134534597'),('1831455690'),('1003203779'),('1477999522'),('1902012180'),('1114360997'),('1104906445'),('1740218296'),('1073711966'),('1588735005'),('1780931956'),('1124319462'),('1215983382')
  AS kol(npi)
),
-- Step 2: Combine your HCP info with KOL list
tagged_hcps AS (
  SELECT
      a.*,
      CASE WHEN b.npi IS NOT NULL THEN 1 ELSE 0 END AS is_kol
  FROM joining_patient_numbers a
  LEFT JOIN kol_list as b
  ON a.npi = b.npi

  UNION ALL

  -- Step 3: Add KOLs missing from your main table
  SELECT
      b.npi,
      NULL AS elaprase_treated_patients,
      NULL AS mpsii_diagnosed_patients,
      NULL AS mpsii_treated_patients_not_elaprase,
      NULL AS mpsii_elaprase_treated_patients,
      1 AS is_kol
  FROM kol_list b
  LEFT JOIN joining_patient_numbers a
  ON a.npi = b.npi
  WHERE a.npi IS NULL
),
hcp_info as (
  select a.*, concat(b.FIRST_NAME, " ", b.LAST_NAME) as hcp_name, b.PRIMARY_SPECIALTY as specialty,
  CASE 
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NULL THEN NULL
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NOT NULL THEN 'Others'
            ELSE c.mapped_bucket
  END AS specialty_bucket, d.HCO_PRIMARY_NPI as hco_npi_v1, d.hco_zipcode as hco_zip_v1, d.hco_primary_name as hco_name_v1
  from tagged_hcps as a
  left join com_edp_prd.com_raw.kom_providers as b on a.npi = b.npi and b.PROVIDER_TYPE = 'INDIVIDUAL'
  left join specialty_groupings as c on b.PRIMARY_SPECIALTY = c.specialty
  left join com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping as d
  on a.npi = d.hcp_npi
),
hcp_vid AS (
  -- get HCP entity VID from vod_hcp so we can join to parent HCO relationships
  SELECT
    TRY_CAST(npi_num__v AS BIGINT) AS hcp_npi,
    vid__v AS hcp_vid
  FROM com_edp_prd.com_raw.vod_hcp
  WHERE npi_num__v IS NOT NULL
),
affiliations AS (
  SELECT
    b.hcp_npi,
    c.PARENT_HCO_VID__V,
    e.npi_num__v as hco_npi_v2,
    e.corporate_name__v as hco_name_v2,
    d.postal_code_cda__v AS hco_zip_v2,
    c.modified_date__v,
    c.status_update_time__v
  FROM hcp_info a
  JOIN hcp_vid b
    ON a.npi = b.hcp_npi
  JOIN com_edp_prd.com_raw.vod_parenthco c
    ON b.hcp_vid = c.ENTITY_VID__V and c.PARENT_HCO_STATUS__V = 'A' AND c.RELATIONSHIP_TYPE__V = '7356' and c.HIERARCHY_TYPE__V = 'HCP_HCO'
  LEFT JOIN com_edp_prd.com_raw.vod_hco e
  ON c.PARENT_HCO_VID__V = e.vid__v
  LEFT JOIN COM_EDP_PRD.COM_RAW.VOD_ADDRESS AS D
    ON c.PARENT_HCO_VID__V = D.ENTITY_VID__V AND D.ENTITY_TYPE__V = 'HCO' 
),
ranked AS (
  SELECT
    hcp_npi,
    hco_npi_v2,
    hco_name_v2,
    hco_zip_v2,
    ROW_NUMBER() OVER (
      PARTITION BY hcp_npi
      ORDER BY modified_date__v DESC NULLS LAST, status_update_time__v DESC NULLS LAST
    ) AS rn
  FROM affiliations
),
top_ranked_hco as (SELECT
  hcp_npi,
  TRY_CAST(hco_npi_v2 AS BIGINT) AS hco_npi_v2,
  hco_name_v2,
  hco_zip_v2
FROM ranked
WHERE rn = 1 and hco_npi_v2 is not null
ORDER BY hcp_npi),
hcp_info_with_hco_info as (
  SELECT
  a.npi, a.elaprase_treated_patients, a.mpsii_diagnosed_patients, a.mpsii_treated_patients_not_elaprase, a.mpsii_elaprase_treated_patients, a.hcp_name, a.specialty, a.specialty_bucket, 
  -- COALESCE(TRY_CAST(a.hco_npi_thm AS BIGINT), b.hco_npi_vod) AS primary_hco, coalesce(a.hco_zip_thm, b.hco_zip) as hco_zipcode, coalesce(a.hco_name_thm, b.hco_name) as hco_name
  case when TRY_CAST(a.hco_npi_v1 AS BIGINT) is not null then TRY_CAST(a.hco_npi_v1 AS BIGINT)
  else b.hco_npi_v2 end as hco_npi,
  case when TRY_CAST(a.hco_npi_v1 AS BIGINT) is not null then hco_name_v1
  else hco_name_v2 end as hco_name,
  case when TRY_CAST(a.hco_npi_v1 AS BIGINT) is not null then hco_zip_v1
  else hco_zip_v2 end as hco_zip
FROM hcp_info a
LEFT JOIN top_ranked_hco b
  ON a.npi = b.hcp_npi
),
hco_type_info_added as (
  select a.*, c.name as hco_type_info
  from hcp_info_with_hco_info as a
  left join com_edp_prd.com_raw.vod_hco as b
on a.hco_npi = try_cast(b.npi_num__v as BIGINT)
left join com_edp_prd.com_raw.vod_references as c
on b.hco_type__v = c.code and c.reference_type = 'HCOType'
),
territory_mapping as (
  select a.*, b.territory_name as hco_territory
  from hco_type_info_added as a
  left join com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping as b
  on a.hco_zip = b.zipcode
)
select * from territory_mapping as a
;